In [ ]:
#This is a follow up to "split_monthly_mat_to_24hour_h5.ipynb"

#This does not embed the annotations into the h5 files. 

#I'm using the "science_env" kernel

#I've created 2 python files:
# F:\Documents\GitHub\ml_development\ADCP_ML\convert_monthly_mat_to_h5.py
#F:\Documents\GitHub\ml_development\ADCP_ML\split_h5_to_24hr_files.py

#I'll want to create a loop that scans the monthly mat folder structure and outputs hourly files
# There are also intermediate monthly h5 files, so will probably be good to clean those up as this runs

In [1]:
#Code to test folder scan

import os
import datetime

data_parent = r'F:\Documents\Projects\ADCP\scan_for_data\BACAX\ADCP2MHZ\\'
folder_list = os.listdir(data_parent)

#I only want folders including and after '20110715', as device is oriented down-facing prior to this date
cutoff_date = datetime.datetime(2011,7,15)
for folder in folder_list:
    print(folder)
    folder_date = datetime.datetime.strptime(folder,'%Y%m%d')
    if folder_date >= cutoff_date:
        file_list = os.listdir(data_parent + folder)
        mat_files = {k for k in file_list if os.path.splitext(k)[1] == ".mat"}
        print(mat_files)


20100517
20100601
20100701
20100801
20100901
20101001
20101101
20101201
20110101
20110201
20110301
20110401
20110501
20110601
20110701
20110715
{'BarkleyCanyon_BarkleyCanyonAxis_AcousticDopplerCurrentProfiler2MHz_20110728T210500Z_20110801T000000Z-Ensemble300s_binMapNearest.mat', 'BarkleyCanyon_BarkleyCanyonAxis_AcousticDopplerCurrentProfiler2MHz_20110720T172000Z_20110728T210500Z-Ensemble300s_binMapNearest.mat', 'BarkleyCanyon_BarkleyCanyonAxis_AcousticDopplerCurrentProfiler2MHz_20110715T195000Z_20110720T172000Z-Ensemble300s_binMapNearest.mat'}
20110801
{'BarkleyCanyon_BarkleyCanyonAxis_AcousticDopplerCurrentProfiler2MHz_20110801T000000Z_20110901T000000Z-Ensemble300s_binMapNearest.mat'}
20110901
{'BarkleyCanyon_BarkleyCanyonAxis_AcousticDopplerCurrentProfiler2MHz_20110901T000000Z_20111001T000000Z-Ensemble300s_binMapNearest.mat'}
20111001
{'BarkleyCanyon_BarkleyCanyonAxis_AcousticDopplerCurrentProfiler2MHz_20111001T000000Z_20111101T000000Z-Ensemble300s_binMapNearest.mat'}
20111101
{'Bark

In [6]:
  #Combine the filenames into a proper path:
mat_paths = []
for filename in mat_files:
    if filename != 'API_record.mat':
        print(filename)
        mat_paths.append(data_parent + folder + '\\' + filename) 


BarkleyCanyon_BarkleyCanyonAxis_AcousticDopplerCurrentProfiler2MHz_20250701T000000Z_20250801T000000Z-Ensemble300s_binMapNearest.mat


In [ ]:
#The function below uses the new "no_embed Data"
# AND
# I've implemented code in "convert_monthly_mat_to_h5.py" and here, that checks if the data has already been converted to h5, 
# so it's not having to repeat all the processing for every new month of data!

In [ ]:
import os
import datetime

# Add repo root to Python path - Needed to import from src folder
import sys
from pathlib import Path
repo_root = Path().resolve().parent  # notebooks/ → ADCP-CNN-QAQC
sys.path.append(str(repo_root))

from src import convert_monthly_mat_to_h5
# from src import split_h5_to_24hr_files_noEmbed
import split_h5_to_24hr_files_noEmbed

import importlib
importlib.reload(split_h5_to_24hr_files_noEmbed)
importlib.reload(convert_monthly_mat_to_h5)

######################################
# Convert mat to h5 (still monthly format)
######################################

data_parent = r'F:\Documents\Projects\ADCP\scan_for_data\BACAX\ADCP2MHZ\\'
folder_list = os.listdir(data_parent)

# Define output folder -  Monthly h5
h5_monthly_folder = r'F:\Documents\Projects\ML\ADCP_ML\BACAX\h5_files\\'

#Output folder - 24hr h5:
h5_24h_folder = r'F:\Documents\Projects\ML\ADCP_ML\BACAX\h5_24h_files_noEmbed\\' 

#I only want folders including and after '20110715', as device is oriented down-facing prior to this date
#I can also adjust this date if the code crashes part way through
cutoff_date = datetime.datetime(2011,7,15)

for folder in folder_list:
    print(folder)
    folder_date = datetime.datetime.strptime(folder,'%Y%m%d')
    if folder_date >= cutoff_date:
        # Path to your .mat file
        file_list = os.listdir(data_parent + folder)
        mat_files = {k for k in file_list if os.path.splitext(k)[1] == ".mat"}
        print(mat_files)

        #Combine the filenames into a proper path:
        mat_paths = []
        for filename in mat_files:
            if filename != 'API_record.mat': # Omit API record files
                mat_paths.append(data_parent + folder + '\\' + filename) 

        #Run the extraction
        for mat_path in mat_paths:
            #print(mat_path)
            save_status = convert_monthly_mat_to_h5.extract_mat_to_h5(mat_path, h5_monthly_folder) 
            #save_status = convert_monthly_mat_to_h5.extract_mat_to_h5(mat_path, h5_monthly_folder, allow_overwrite = 1) # To allow overwriting

        #Only proceed with splitting files if the monthly h5 file didn't already exist.  Otherwise, likely re-extracting
        #This is so that this can be efficiently run on new files, without having to re-run for ALL data
        if save_status == 'success':
            ######################################
            # Split to 24 hours, 
            # and save the time in python format
            ######################################
                
            #Paths to month(ish) HDF5 source file(s):
            for mat_path in mat_paths:
                filename_mat = os.path.basename(mat_path)
                filename_h5 = os.path.splitext(filename_mat)[0] + '.h5'
                input_file = h5_monthly_folder + filename_h5

                print('Splitting files from folder {}'.format(folder))

                split_h5_to_24hr_files_noEmbed.split_h5_to_24hr_files(
                    input_file,             # your big HDF5 source (created with import_monthly_mat_to_h5.py)
                    h5_24h_folder,          # output dir for 24hr files
                )

######################################
#Test code to extract and plot the data 
######################################
#filename = r'F:\\Documents\\GitHub\\ml_development\\ADCP_ML\\h5_24h_files\\20240406T000000_20240406T235959.h5'
#make_sanity_plots(filename, outdir="./h5_24hr_figs", show=True)

20100517
20100601
20100701
20100801
20100901
20101001
20101101
20101201
20110101
20110201
20110301
20110401
20110501
20110601
20110701
20110715
{'BarkleyCanyon_BarkleyCanyonAxis_AcousticDopplerCurrentProfiler2MHz_20110728T210500Z_20110801T000000Z-Ensemble300s_binMapNearest.mat', 'BarkleyCanyon_BarkleyCanyonAxis_AcousticDopplerCurrentProfiler2MHz_20110720T172000Z_20110728T210500Z-Ensemble300s_binMapNearest.mat', 'BarkleyCanyon_BarkleyCanyonAxis_AcousticDopplerCurrentProfiler2MHz_20110715T195000Z_20110720T172000Z-Ensemble300s_binMapNearest.mat'}
Skipping (already exists): F:\Documents\Projects\ML\ADCP_ML\h5_files\\BarkleyCanyon_BarkleyCanyonAxis_AcousticDopplerCurrentProfiler2MHz_20110728T210500Z_20110801T000000Z-Ensemble300s_binMapNearest.h5
Skipping (already exists): F:\Documents\Projects\ML\ADCP_ML\h5_files\\BarkleyCanyon_BarkleyCanyonAxis_AcousticDopplerCurrentProfiler2MHz_20110720T172000Z_20110728T210500Z-Ensemble300s_binMapNearest.h5
Skipping (already exists): F:\Documents\Projects

In [ ]:
######################################
#Test code to extract and plot the data 
######################################
#filename = r'F:\\Documents\\GitHub\\ml_development\\ADCP_ML\\h5_24h_files\\20240406T000000_20240406T235959.h5'
#make_sanity_plots(filename, outdir="./h5_24hr_figs", show=True)